# Early-fusion baseline

**One sentence:** given an RNA window + an RBP, predict whether that RBP binds that window, using the simplest possible model — `concat(RNA_vec, protein_vec) -> MLP -> sigmoid`.

**Why:** sanity-check baseline for the team's multimodal pipeline. If this can't learn anything above chance, the data/labels/mapping are broken and no fancy model will help. If it learns something, FiLM and cross-attention have a number to beat.

**Outline:**
1. **Probe the data** — verify schema, shapes, label range (don't trust assumptions)
2. **Build gene → ProtT5 mapping** — check coverage of the dataset's RBPs
3. **Construct datasets** — one example per (window, RBP) pair
4. **Train** — MLP, BCE loss, AdamW, few epochs
5. **Evaluate** — AUROC + AUPRC on the validation split
6. **Controls** — protein-shuffle + RNA-only (mandatory; see CONTRACT.md)

**Where things live:**
- Data: `~/storage_ml4rg26-mmparnet/manually_gathered/` (on the VM)
- Model + loaders: `src/mmpartnet/baselines/early_fusion.py`
- This notebook: orchestration only; if you find yourself writing model code here, move it into the module.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

# make the package importable when running from notebooks/
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from mmpartnet.baselines.early_fusion import (
    EarlyFusion, EarlyFusionDataset, ProteinEmbeddings,
    build_gene_to_idx, train_one_epoch, evaluate, protein_shuffle_mapping,
)

# data paths (on the VM)
BASE = Path("~/storage_ml4rg26-mmparnet/manually_gathered").expanduser()
DATASET_DIR = BASE / "600nt_windows.no-one-hot.stripped.binding" / "600nt_windows.no-one-hot.stripped.binding.narrowpeak_intersect"
DATASET_PT  = DATASET_DIR / "dataset.pt"
RBP_CTS_TSV = DATASET_DIR / "rbp_cts.tsv"
PROTT5_DIR  = BASE / "ProtT5_zenodo_datasets"
PROTT5_H5   = PROTT5_DIR / "reduced_embeddings_file.h5"
HUMAN_FASTA = PROTT5_DIR / "human.fasta"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={DEVICE}")
print(f"repo={REPO}")

device=cpu
repo=/mnt/storage1/workspace/cgerards/ML4RG_mmparnet


## Step 1 — probe the data

Confirm the on-disk layout before writing any training code. Bugs here are **silent and catastrophic**: if the binding tensor columns don't match the RBP order in `rbp_cts.tsv`, we'd train on shuffled labels and never know.

What we expect (from earlier probes):
- `dataset.pt` is a dict with `train` / `test` / `valid` lists of records
- Each record is `{inputs: {sequence: str}, outputs: {binding: tensor}, meta: {name, pad_side}}`
- `binding.shape[0]` should equal `len(rbp_cts.tsv)`

The `assert` below will fail loudly if that alignment is wrong.

In [2]:
# Loading takes ~4 minutes on the first call (large pickle). Cached for kernel lifetime.
if "d" not in globals():
    d = torch.load(DATASET_PT)

print(f"splits: { {k: len(v) for k, v in d.items()} }")

# inspect one record
rec = d["train"][0]
seq = rec["inputs"]["sequence"]
binding = rec["outputs"]["binding"]
print(f"sequence length: {len(seq)}")
print(f"binding tensor: shape={tuple(binding.shape)} dtype={binding.dtype} sum(positives)={int(binding.sum())}")
print(f"meta: {rec['meta']}")

# RBP column order
rbp_cts = pd.read_csv(RBP_CTS_TSV, sep="\t")
print(f"\nrbp_cts.tsv shape: {rbp_cts.shape}")
print(f"columns: {list(rbp_cts.columns)}")
print(rbp_cts.head())

# pick the column that holds the RBP symbol — usually 'rbp' or the first column
RBP_COL = "rbp" if "rbp" in rbp_cts.columns else rbp_cts.columns[0]
rbps = rbp_cts[RBP_COL].tolist()
print(f"\nN RBPs from tsv: {len(rbps)}, first 5: {rbps[:5]}")

assert len(rbps) == binding.shape[0], (
    f"RBP count mismatch! tsv={len(rbps)} vs binding={binding.shape[0]} — labels would be misaligned"
)
print("OK: rbp_cts rows match binding tensor length")

splits: {'train': 512946, 'test': 70626, 'valid': 116542}
sequence length: 600
binding tensor: shape=(223,) dtype=torch.float32 sum(positives)=1
meta: {'name': 'chr14:100374289-100374889:-', 'pad_side': -1}

rbp_cts.tsv shape: (223, 3)
columns: ['rbp_ct', 'rbp', 'ct']
        rbp_ct    rbp     ct
0    AARS_K562   AARS   K562
1    AATF_K562   AATF   K562
2   ABCF1_K562  ABCF1   K562
3  AGGF1_HepG2  AGGF1  HepG2
4   AGGF1_K562  AGGF1   K562

N RBPs from tsv: 223, first 5: ['AARS', 'AATF', 'ABCF1', 'AGGF1', 'AGGF1']
OK: rbp_cts rows match binding tensor length


## Step 2 — build the gene → ProtT5 mapping and check coverage

The ProtT5 h5 keys are **sequential indices into `human.fasta`** (`'0'`, `'1'`, ...). UniProt fasta headers carry the gene symbol in `GN=<symbol>`. We parse the fasta once, build `{gene: h5_key}`, then check how many of our dataset's RBPs have a ProtT5 vector.

**Anything missing means we can't use that RBP in training** — we drop it from the RBP subset. If coverage is bad (<80%), stop and investigate before training (likely the fasta uses a different symbol convention, or the dataset RBPs are non-human or aliases).

In [3]:
gene_to_idx = build_gene_to_idx(HUMAN_FASTA)
print(f"parsed {len(gene_to_idx)} gene->idx mappings from fasta")

covered = [g for g in rbps if g in gene_to_idx]
missing = [g for g in rbps if g not in gene_to_idx]
print(f"coverage: {len(covered)}/{len(rbps)} RBPs have a ProtT5 vector")
if missing:
    print(f"missing ({len(missing)}): {missing[:20]}{' ...' if len(missing) > 20 else ''}")

proteins = ProteinEmbeddings(PROTT5_H5, gene_to_idx)

# sanity check: fetch one
v = proteins.vector(covered[0])
print(f"\nexample: protein vector for {covered[0]!r} -> shape={v.shape}, dtype={v.dtype}")

parsed 20175 gene->idx mappings from fasta
coverage: 220/223 RBPs have a ProtT5 vector
missing (3): ['AARS', 'TROVE2', 'TROVE2']

example: protein vector for 'AATF' -> shape=(1024,), dtype=float32


## Step 3 — construct datasets

One training example = one `(window, RBP)` pair. The full train set is **huge**:
`512,946 windows × ~200 RBPs ≈ 100M pairs`. We subsample windows for a first run (still hundreds of thousands of pairs — plenty to learn from). Ramp this up later once you trust the pipeline.

In [4]:
# Tune these to control training set size. (200 RBPs * N_TRAIN_WINDOWS = N pairs.)
N_TRAIN_WINDOWS = 5000     # of 512,946
N_VALID_WINDOWS = 2000     # of 116,542

rng = np.random.default_rng(0)
train_idx = rng.choice(len(d["train"]), N_TRAIN_WINDOWS, replace=False).tolist()
valid_idx = rng.choice(len(d["valid"]), N_VALID_WINDOWS, replace=False).tolist()

train_records = [d["train"][i] for i in train_idx]
valid_records = [d["valid"][i] for i in valid_idx]

train_ds = EarlyFusionDataset(train_records, rbps, proteins, rbp_subset=covered)
valid_ds = EarlyFusionDataset(valid_records, rbps, proteins, rbp_subset=covered)
print(f"train pairs: {len(train_ds):,}   valid pairs: {len(valid_ds):,}")
print(f"RBPs in subset: {train_ds.N_rbps}")

# spot-check class balance on a small slice (positives are sparse — expect ~1-5%)
sample_size = min(5000, len(train_ds))
sample_ix = rng.choice(len(train_ds), sample_size, replace=False).tolist()
sample_labels = np.array([train_ds[i][2].item() for i in sample_ix])
pos_rate = float(sample_labels.mean())
print(f"positive rate (sampled n={sample_size}): {pos_rate:.4f}")

train pairs: 1,100,000   valid pairs: 440,000
RBPs in subset: 220
positive rate (sampled n=5000): 0.0042


## Step 4 — train

A 2-hidden-layer MLP on the concatenated `[RNA(4-d), ProtT5(1024-d)]` vector. Few epochs is enough to see if it learns at all.

**Class imbalance**: positives are sparse (~few %), so we weight them via BCE `pos_weight`. Without this, the model converges to predicting all-zeros and gets near-perfect loss while being useless.

In [5]:
BATCH = 512
EPOCHS = 3
LR = 1e-4

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, persistent_workers=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH, shuffle=False, num_workers=2, persistent_workers=True)

model = EarlyFusion(rna_dim=4, prot_dim=1024).to(DEVICE)
optim = torch.optim.AdamW(model.parameters(), lr=LR)

pos_weight = torch.tensor((1 - pos_rate) / max(pos_rate, 1e-6), device=DEVICE)
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

for epoch in range(EPOCHS):
    tr_loss = train_one_epoch(model, train_loader, optim, loss_fn, DEVICE)
    m = evaluate(model, valid_loader, DEVICE)
    print(f"epoch {epoch+1}: train_loss={tr_loss:.4f}  val_auroc={m['auroc']:.4f}  val_auprc={m['auprc']:.4f}  pos_rate={m['pos_rate']:.4f}")

metrics = m

epoch 1: train_loss=1.2678  val_auroc=0.8097  val_auprc=0.0220  pos_rate=0.0051
epoch 2: train_loss=1.1114  val_auroc=0.8220  val_auprc=0.0252  pos_rate=0.0051
epoch 3: train_loss=1.0752  val_auroc=0.8276  val_auprc=0.0269  pos_rate=0.0051


## Step 5 — controls (MANDATORY)

Every number reported must come with both controls (per `CONTRACT.md`):

1. **Protein-shuffle:** permute the gene → ProtT5 mapping. If AUROC barely drops, the model is **not using** the protein channel — the result is fake.
2. **RNA-only ablation:** zero out the protein vector at eval time. AUROC should drop noticeably; if the full model isn't above RNA-only, the protein adds nothing.

Verdict heuristic: `full - shuffle > 0.02` AND `full - rna_only > 0.02` → protein is contributing.

In [6]:
# Control 1: protein-shuffle
shuffled = protein_shuffle_mapping(gene_to_idx, seed=42)
proteins_shuf = ProteinEmbeddings(PROTT5_H5, shuffled)
valid_ds_shuf = EarlyFusionDataset(valid_records, rbps, proteins_shuf, rbp_subset=covered)
valid_loader_shuf = DataLoader(valid_ds_shuf, batch_size=BATCH, shuffle=False, num_workers=2)
metrics_shuf = evaluate(model, valid_loader_shuf, DEVICE)

# Control 2: RNA-only (zero the protein channel at eval time)
@torch.no_grad()
def evaluate_rna_only(model, loader, device):
    from sklearn.metrics import roc_auc_score, average_precision_score
    model.eval()
    ys, ps = [], []
    for rna, prot, y in loader:
        rna = rna.to(device)
        prot_zero = torch.zeros_like(prot).to(device)
        p = torch.sigmoid(model(rna, prot_zero)).cpu().numpy()
        ys.append(y.numpy()); ps.append(p)
    y = np.concatenate(ys); p = np.concatenate(ps)
    return {"auroc": float(roc_auc_score(y, p)), "auprc": float(average_precision_score(y, p))}

metrics_rna = evaluate_rna_only(model, valid_loader, DEVICE)

print(f"full model:         val_auroc={metrics['auroc']:.4f}   val_auprc={metrics['auprc']:.4f}")
print(f"protein-shuffle:    val_auroc={metrics_shuf['auroc']:.4f}   val_auprc={metrics_shuf['auprc']:.4f}")
print(f"RNA-only (zeroed):  val_auroc={metrics_rna['auroc']:.4f}   val_auprc={metrics_rna['auprc']:.4f}")

delta_shuf = metrics['auroc'] - metrics_shuf['auroc']
delta_rna  = metrics['auroc'] - metrics_rna['auroc']
print(f"\nΔ vs shuffle:   {delta_shuf:+.4f}")
print(f"Δ vs RNA-only:  {delta_rna:+.4f}")

if delta_shuf > 0.02 and delta_rna > 0.02:
    print("\nVERDICT: the protein channel is contributing -> baseline is real.")
else:
    print("\nVERDICT: protein channel is not pulling its weight. Investigate before reporting.")

full model:         val_auroc=0.8276   val_auprc=0.0269
protein-shuffle:    val_auroc=0.5499   val_auprc=0.0101
RNA-only (zeroed):  val_auroc=0.6272   val_auprc=0.0101

Δ vs shuffle:   +0.2777
Δ vs RNA-only:  +0.2004

VERDICT: the protein channel is contributing -> baseline is real.


## Where to go next

Once this notebook produces a real (controlled) AUROC:

1. **Better RNA encoder** — swap `one_hot_meanpool` for PARNET body features (`mmpartnet.models.parnet.load_parnet().body_feats(...)` mean-pooled to 512-d). One change in `EarlyFusionDataset(..., rna_encoder=...)`.
2. **More training data** — increase `N_TRAIN_WINDOWS`; eventually use the full split.
3. **Better negative sampling** — the current loader treats every non-positive (window, RBP) as a negative. The team's contract calls for **RNA-matched hard-N2** negatives; that's a follow-up.
4. **Compare to fancier fusion** — once your teammate's FiLM head lands, run both on the same splits and report the gap.

Anything more complex than this notebook belongs in the module (`src/mmpartnet/baselines/early_fusion.py`) or a new module under `baselines/`.